<a href="https://colab.research.google.com/github/R-Bhargavsai/Predict-credit-card-approvals.ipynb/blob/main/Predict_credit_card_approvals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Credit Card Approval Prediction Using Logistic Regression

## 1. Project Overview & Exploratory Data Analysis
This notebook aims to predict credit card approvals using a dataset containing financial indicators and credit history metrics. We will implement data preprocessing, cross-validation hyperparameter tuning, and evaluate performance using classification metrics.

### Key Step Objectives:
* Inspect the structured data schema and distributions.
* Identify missing or anomalies markers (`?`).
* Assess descriptive summary statistics (`count`, `unique`, `top`, `freq`).

In [ ]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import MinMaxScaler

# Load dataset (Replace 'credit_data.csv' with your actual file path)
# Assuming a standard UCI format where missing values are marked with '?'
df = pd.read_csv("/content/datalab_export_2026-05-28 08_49_55.csv", header=None, na_values="?")

# Inspect the first few rows
print("--- Dataset Preview ---")
print(df.head())

# Print summary statistics
print("\n--- Summary Statistics ---")
print(df.describe())

--- Dataset Preview ---
      0              1                   2         3            4   \
0  index  credit.policy             purpose  int.rate  installment   
1      0              1  debt_consolidation    0.1189        829.1   
2      1              1         credit_card    0.1071       228.22   
3      2              1  debt_consolidation    0.1357       366.86   
4      3              1  debt_consolidation    0.1008       162.34   

               5      6     7                  8          9           10  \
0  log.annual.inc    dti  fico  days.with.cr.line  revol.bal  revol.util   
1     11.35040654  19.48   737        5639.958333      28854        52.1   
2     11.08214255  14.29   707               2760      33623        76.7   
3     10.37349118  11.63   682               4710       3511        25.6   
4     11.35040654    8.1   712        2699.958333      33667        73.2   

               11           12       13              14  
0  inq.last.6mths  delinq.2yrs  pub.rec 

## 2. Target Variable Analysis & Feature Dropping

In this phase, we isolate our predictive features ($X$) from our target label ($y$). The index column is removed as it holds no predictive weight.

### Key Observations:
* **Initial Features:** 13 predictive variables.
* **Severe Class Imbalance:** The target variable `not.fully.paid` contains **90** instances of `0` (Paid) and only **10** instances of `1` (Not fully paid).
* **Implication:** Accuracy will likely be a deceptive metric. A naive model predicting `0` for everything would automatically achieve 90% accuracy.

In [ ]:
df = pd.read_csv("datalab_export_2026-05-28 08_49_55.csv")


if "index" in df.columns:
    df = df.drop(columns=["index"])

X = df.drop(columns=["not.fully.paid"])
y = df["not.fully.paid"]

print("Initial data dimensions:", X.shape)
print("Target label distribution:\n", y.value_counts())

Initial data dimensions: (100, 13)
Target label distribution:
 not.fully.paid
0    90
1    10
Name: count, dtype: int64


## 3. Handling Missing Values (Imputation)

Real-world datasets often suffer from missing features. Dropping entire rows destroys valuable data, so we deploy a strategic mathematical imputation approach:

* **Numeric Features:** Imputed using the **mean** strategy to preserve the overall distribution average.
* **Categorical Features:** Imputed using the **most frequent (mode)** strategy to preserve categorical popularity without introducing artificial text.

In [ ]:
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()


num_imputer = SimpleImputer(strategy="mean")
X[numeric_cols] = num_imputer.fit_transform(X[numeric_cols])

if categorical_cols:
    cat_imputer = SimpleImputer(strategy="most_frequent")
    X[categorical_cols] = cat_imputer.fit_transform(X[categorical_cols])

## 4. One-Hot Encoding Categorical Variables

Machine learning models require mathematical matrix inputs. To process categorical text values, we perform **One-Hot Encoding**.

### Configuration Notes:
* `drop_first=True` is utilized to prevent **multicollinearity** (the dummy variable trap) by dropping one redundant category column per feature.
* **Dimensionality Expansion:** Encoding increased our feature count from 13 to 18 columns.

In [ ]:
X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)
print("Dimensions after encoding categorical variables:", X.shape)

Dimensions after encoding categorical variables: (100, 18)


## 5. Dataset Splitting & Feature Scaling

### Strategic Splitting:
* **Test Size:** 30% of data is reserved for unseen evaluation (`X_test`), while 70% trains the model (`X_train`).
* **Stratification (`stratify=y`):** Crucial due to the extreme class imbalance. It ensures both train and test splits preserve the exact 90:10 ratio of target classes.

### Normalization (`MinMaxScaler`):
* Scales all numeric variables precisely between 0 and 1.
* Prevents columns with large magnitudes (e.g., `revol.bal` or annual income) from over-influencing the Logistic Regression coefficients compared to smaller ranges (e.g., `int.rate`).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

## 6. Hyperparameter Optimization via Grid Search

To maximize model performance, we use 5-fold cross-validation (`GridSearchCV`) to test multiple hyperparameter permutations.

### Grid Parameters Explored:
* `C` (Regularization strength): Controls overfitting by penalizing large weights. Lesser values mean stronger regularization.
* `class_weight`: Tested `None` vs. `balanced` to penalize minority class mistakes heavily.
* `scoring='f1_weighted'`: Guided by weighted F1-score rather than accuracy to explicitly counter the data imbalance challenge.

In [ ]:
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 7. Model Diagnostic Summary & Next Steps

### Results Analysis:
* **Optimal Params Selected:** `{'C': 0.01, 'class_weight': None, 'tol': 0.01}`
* **The Dummy Classifier Trap:** Despite a flashy **90.00% Accuracy Score**, the detailed metrics reveal a major flaw. The confusion matrix shows `[[27, 0], [3, 0]]`.
* **Severe Minority Failure:** The model predicted class `0` for 100% of the cases. Consequently, for class `1` (Not fully paid), the **Precision, Recall, and F1-score are all 0.00**.

### Technical Takeaways & Recommendations:
1.  **Dataset Volume Constraint:** The dataset contains only 100 records. Machine learning algorithms struggle to discover viable boundaries with only 10 positive class samples.
2.  **Imbalance Mitigation:** Since `class_weight='balanced'` was rejected by the grid search in favor of a heavy $C$ penalty (`C=0.01`), future iterations should try oversampling techniques like **SMOTE** or using tree-based architectures (e.g., Random Forests).
3.  **Data Expansion:** It is strongly recommended to acquire more data observations before rolling this model out to production.

In [ ]:
log_reg = LogisticRegression(max_iter=1000)


param_grid = {
    "C": [0.01, 0.1, 1.0, 10.0, 100.0],
    "tol": [0.01, 0.001, 0.0001],
    "class_weight": [None, "balanced"],
}

grid_search = GridSearchCV(
    estimator=log_reg, param_grid=param_grid, cv=5, scoring="f1_weighted"
)
grid_search.fit(X_train_scaled, y_train)

GridSearchCV(cv=5, estimator=LogisticRegression(max_iter=1000),
             param_grid={'C': [0.01, 0.1, 1.0, 10.0, 100.0],
                         'class_weight': [None, 'balanced'],
                         'tol': [0.01, 0.001, 0.0001]},
             scoring='f1_weighted')

In [ ]:
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test_scaled)

print("\n--- Model Tuning Optimization Results ---")
print("Best Hyperparameters Chosen:", grid_search.best_params_)
print(f"Best Weighted $F_1$ Cross-Validation Score: {grid_search.best_score_:.4f}")

print("\n--- Unseen Test Set Metrics ---")
print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nDetailed Classification Report:")
print(classification_report(y_test, y_pred))


--- Model Tuning Optimization Results ---
Best Hyperparameters Chosen: {'C': 0.01, 'class_weight': None, 'tol': 0.01}
Best Weighted $F_1$ Cross-Validation Score: 0.8530

--- Unseen Test Set Metrics ---
Accuracy Score: 0.9000

Confusion Matrix:
[[27  0]
 [ 3  0]]

Detailed Classification Report:
              precision    recall  f1-score   support

           0       0.90      1.00      0.95        27
           1       0.00      0.00      0.00         3

    accuracy                           0.90        30
   macro avg       0.45      0.50      0.47        30
weighted avg       0.81      0.90      0.85        30



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
